# Round 4 — Trader Markov Models

We model each counterparty as a **discrete-time Markov chain**.  
Each trade is a step; the *state* is `(product, side)` — e.g. `(HYDROGEL_PACK, buy)`.  
Transition matrices tell us: *given trader X just did A, what will they do next?*

## 1 — Setup & Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import chi2_contingency
from pathlib import Path

DATA_DIR = Path("../../historical_data/round_4")

frames = []
for d in [1, 2, 3]:
    tmp = pd.read_csv(DATA_DIR / f"trades_round_4_day_{d}.csv", sep=";")
    tmp["day"] = d
    frames.append(tmp)
trades = pd.concat(frames, ignore_index=True)

# Build global timestamp
ticks_per_day = 1_000_000
trades["global_ts"] = (trades["day"] - 1) * ticks_per_day + trades["timestamp"]

# Expand to long form: one row per (trade, trader_side)
buy_rows  = trades[["global_ts","day","timestamp","buyer", "symbol","price","quantity"]].copy()
sell_rows = trades[["global_ts","day","timestamp","seller","symbol","price","quantity"]].copy()
buy_rows.columns  = ["global_ts","day","timestamp","trader","product","price","quantity"]
sell_rows.columns = ["global_ts","day","timestamp","trader","product","price","quantity"]
buy_rows["side"]  = "buy"
sell_rows["side"] = "sell"

activity = pd.concat([buy_rows, sell_rows], ignore_index=True)
activity = activity[activity["trader"].notna()].copy()
activity = activity.sort_values(["trader","global_ts"]).reset_index(drop=True)

# State = (product, side)
activity["state"] = activity["product"] + "|" + activity["side"]

TRADERS  = sorted(activity["trader"].unique())
PRODUCTS = sorted(activity["product"].unique())
SIDES    = ["buy", "sell"]
ALL_STATES = [f"{p}|{s}" for p in PRODUCTS for s in SIDES]

print(f"Total activity rows: {len(activity):,}")
print(f"Traders : {TRADERS}")
print(f"Products: {PRODUCTS}")
print()
print(activity.groupby(["trader","side"]).size().unstack(fill_value=0))

## 2 — Helper: Build Transition Matrix

In [ ]:
def build_transition_matrix(seq, states):
    """
    Build a row-normalised transition matrix from a sequence of state labels.
    Returns (matrix as DataFrame, raw counts DataFrame).
    """
    idx = {s: i for i, s in enumerate(states)}
    n = len(states)
    counts = np.zeros((n, n), dtype=float)
    for a, b in zip(seq[:-1], seq[1:]):
        if a in idx and b in idx:
            counts[idx[a], idx[b]] += 1
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # avoid /0
    probs = counts / row_sums
    return (pd.DataFrame(probs, index=states, columns=states),
            pd.DataFrame(counts, index=states, columns=states))


def stationary_dist(P):
    """
    Compute stationary distribution of a stochastic matrix P (row-normalised).
    Uses eigen-decomposition of P^T.
    """
    vals, vecs = np.linalg.eig(P.values.T)
    # eigenvector for eigenvalue closest to 1
    idx = np.argmin(np.abs(vals - 1.0))
    v = np.real(vecs[:, idx])
    v = np.abs(v)
    return pd.Series(v / v.sum(), index=P.index)


def chi2_test(counts_df):
    """
    Chi-squared test of independence on the counts matrix.
    Returns (chi2_stat, p_value). Small p → transitions are NOT random.
    """
    mat = counts_df.values
    # keep only rows/cols with any counts
    mask = mat.sum(axis=1) > 0
    mat = mat[mask][:, mask]
    if mat.shape[0] < 2:
        return None, None
    try:
        chi2, p, _, _ = chi2_contingency(mat)
        return chi2, p
    except Exception:
        return None, None


print("Helpers loaded.")

## 3 — Direction Markov (Buy / Sell, 2 states)

The simplest model: does each trader switch direction or stay consistent?

In [ ]:
SIDE_STATES = ["buy", "sell"]

dir_results = {}
for trader in TRADERS:
    seq = activity[activity["trader"] == trader]["side"].tolist()
    P, C = build_transition_matrix(seq, SIDE_STATES)
    chi2, p = chi2_test(C)
    pi = stationary_dist(P)
    dir_results[trader] = {"P": P, "C": C, "chi2": chi2, "p": p, "pi": pi, "n": len(seq)}

# Summary table
rows = []
for trader, res in dir_results.items():
    P = res["P"]
    rows.append({
        "trader":         trader,
        "n_trades":       res["n"],
        "P(buy|buy)":     round(P.loc["buy",  "buy"],  3),
        "P(sell|sell)":   round(P.loc["sell", "sell"], 3),
        "pi_buy":         round(res["pi"]["buy"], 3),
        "pi_sell":        round(res["pi"]["sell"], 3),
        "chi2":           round(res["chi2"], 2) if res["chi2"] else "n/a",
        "p_value":        f"{res['p']:.4f}" if res["p"] else "n/a",
    })
dir_summary = pd.DataFrame(rows).set_index("trader")
print(dir_summary.to_string())

In [ ]:
# Visualise 2×2 transition matrices side by side
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
cmap = LinearSegmentedColormap.from_list("wbl", ["white", "#2166AC"])

for ax, trader in zip(axes, TRADERS):
    P = dir_results[trader]["P"]
    im = ax.imshow(P.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks([0, 1]); ax.set_xticklabels(SIDE_STATES, fontsize=9)
    ax.set_yticks([0, 1]); ax.set_yticklabels(SIDE_STATES, fontsize=9)
    ax.set_xlabel("Next state", fontsize=8); ax.set_ylabel("Current state", fontsize=8)
    for i in range(2):
        for j in range(2):
            v = P.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=12,
                    color="white" if v > 0.6 else "black", fontweight="bold")
    n = dir_results[trader]["n"]
    p_val = dir_results[trader]["p"]
    p_str = f"p={p_val:.4f}" if p_val else ""
    ax.set_title(f"{trader}  (n={n})\n{p_str}", fontsize=9)

for j in range(len(TRADERS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Direction Markov — P(next side | current side)", fontsize=13)
plt.tight_layout()
plt.show()

## 4 — Product Markov (12 states)

Given trader X just traded product A, what product will they trade next?

In [ ]:
prod_results = {}
for trader in TRADERS:
    seq = activity[activity["trader"] == trader]["product"].tolist()
    P, C = build_transition_matrix(seq, PRODUCTS)
    chi2, p = chi2_test(C)
    pi = stationary_dist(P)
    prod_results[trader] = {"P": P, "C": C, "chi2": chi2, "p": p, "pi": pi}

# Plot product transition heatmaps
short = {p: p.replace("VELVETFRUIT_EXTRACT", "VEX").replace("HYDROGEL_PACK", "HP")
         for p in PRODUCTS}
labels = [short[p] for p in PRODUCTS]

ncols = 4
nrows = int(np.ceil(len(TRADERS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 5.5))
axes = axes.flatten()

for ax, trader in zip(axes, TRADERS):
    P = prod_results[trader]["P"]
    im = ax.imshow(P.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(PRODUCTS)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=6)
    ax.set_yticks(range(len(PRODUCTS)))
    ax.set_yticklabels(labels, fontsize=6)
    for i in range(len(PRODUCTS)):
        for j in range(len(PRODUCTS)):
            v = P.values[i, j]
            if v > 0.05:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=5,
                        color="white" if v > 0.6 else "black")
    p_val = prod_results[trader]["p"]
    p_str = f"p={p_val:.2e}" if p_val else ""
    ax.set_title(f"{trader}  {p_str}", fontsize=9)
    ax.set_xlabel("Next product", fontsize=7)
    ax.set_ylabel("Current product", fontsize=7)

for j in range(len(TRADERS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Product Markov — P(next product | current product)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5 — Combined (Product × Side) Markov

24-state chain — only shown for traders with enough data (n > 200).

In [ ]:
combined_results = {}
for trader in TRADERS:
    tdf = activity[activity["trader"] == trader]
    if len(tdf) < 100:
        continue
    seq = tdf["state"].tolist()
    # only keep states that actually appear
    active_states = [s for s in ALL_STATES if s in set(seq)]
    P, C = build_transition_matrix(seq, active_states)
    chi2, p = chi2_test(C)
    pi = stationary_dist(P)
    combined_results[trader] = {"P": P, "C": C, "chi2": chi2, "p": p, "pi": pi,
                                 "states": active_states}

for trader, res in combined_results.items():
    P = res["P"]
    states = res["states"]
    s_labels = [s.replace("VELVETFRUIT_EXTRACT","VEX").replace("HYDROGEL_PACK","HP") for s in states]

    fig, ax = plt.subplots(figsize=(max(10, len(states)*0.7), max(8, len(states)*0.65)))
    im = ax.imshow(P.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(states)))
    ax.set_xticklabels(s_labels, rotation=55, ha="right", fontsize=7)
    ax.set_yticks(range(len(states)))
    ax.set_yticklabels(s_labels, fontsize=7)
    for i in range(len(states)):
        for j in range(len(states)):
            v = P.values[i, j]
            if v > 0.05:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if v > 0.55 else "black")
    plt.colorbar(im, ax=ax, label="Transition probability")
    p_val = res["p"]
    p_str = f"p={p_val:.2e}" if p_val else ""
    ax.set_title(f"{trader} — Combined (Product × Side) Markov  {p_str}", fontsize=11)
    ax.set_xlabel("Next state"); ax.set_ylabel("Current state")
    plt.tight_layout()
    plt.show()

    # top-5 most likely transitions
    flat = P.stack().reset_index()
    flat.columns = ["from", "to", "prob"]
    flat = flat[flat["from"] != flat["to"]].sort_values("prob", ascending=False)
    print(f"\n{trader} — top transitions:")
    print(flat.head(10).to_string(index=False))
    print()

## 6 — Stationary Distributions

The long-run fraction of time each trader spends in each (product, side) state.

In [ ]:
fig, axes = plt.subplots(len(combined_results), 1,
                          figsize=(16, len(combined_results) * 3.2))
if len(combined_results) == 1:
    axes = [axes]

for ax, (trader, res) in zip(axes, combined_results.items()):
    pi = res["pi"].sort_values(ascending=False)
    colors = ["#55A868" if "|buy" in s else "#C44E52" for s in pi.index]
    x_labels = [s.replace("VELVETFRUIT_EXTRACT","VEX").replace("HYDROGEL_PACK","HP") for s in pi.index]
    bars = ax.bar(range(len(pi)), pi.values, color=colors, alpha=0.85)
    ax.set_xticks(range(len(pi)))
    ax.set_xticklabels(x_labels, rotation=50, ha="right", fontsize=7)
    ax.set_ylabel("Stationary prob.", fontsize=8)
    ax.set_title(f"{trader} — Stationary Distribution", fontsize=10)
    ax.grid(True, alpha=0.3, axis="y")
    green_patch = plt.Rectangle((0,0),1,1, color="#55A868", alpha=0.85, label="buy")
    red_patch   = plt.Rectangle((0,0),1,1, color="#C44E52", alpha=0.85, label="sell")
    ax.legend(handles=[green_patch, red_patch], fontsize=8)

fig.suptitle("Long-Run Stationary Distributions per Trader", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7 — Self-Transition (Persistence) Analysis

How strongly does each trader repeat their last action?  
High diagonal → highly predictable / persistent behaviour.

In [ ]:
# For direction chain: P(same side | current side)
rows = []
for trader in TRADERS:
    P  = dir_results[trader]["P"]
    seq = activity[activity["trader"] == trader]["side"].tolist()
    n_buy  = seq.count("buy")
    n_sell = seq.count("sell")
    rows.append({
        "trader":       trader,
        "n":            len(seq),
        "pct_buy":      100 * n_buy / len(seq),
        "P(buy|buy)":   P.loc["buy",  "buy"],
        "P(sell|sell)": P.loc["sell", "sell"],
        "mean_persist": (P.loc["buy","buy"] * n_buy + P.loc["sell","sell"] * n_sell) / len(seq),
    })

persist_df = pd.DataFrame(rows).set_index("trader")
print(persist_df.round(3).to_string())

# Bar chart of direction persistence
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(TRADERS))
w = 0.28
ax.bar(x - w, persist_df["pct_buy"],       w, label="% of trades as buyer",  color="#4C72B0", alpha=0.85)
ax.bar(x,     persist_df["P(buy|buy)"] * 100, w, label="P(buy|buy) ×100",    color="#55A868", alpha=0.85)
ax.bar(x + w, persist_df["P(sell|sell)"] * 100, w, label="P(sell|sell) ×100", color="#C44E52", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(TRADERS, fontsize=9)
ax.axhline(50, color="grey", ls="--", lw=0.8, alpha=0.7, label="50 %")
ax.set_ylabel("%")
ax.set_title("Directional Persistence by Trader", fontsize=12)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 8 — Time Between Trades (Waiting-Time Distribution)

How quickly does each trader place their next trade after the previous one?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for ax, trader in zip(axes, TRADERS):
    tdf = activity[activity["trader"] == trader].sort_values("global_ts")
    gaps = tdf["global_ts"].diff().dropna()
    gaps = gaps[gaps > 0]  # ignore simultaneous bundle trades
    ax.hist(gaps, bins=50, color="#4C72B0", alpha=0.8, edgecolor="white", lw=0.3)
    ax.set_title(f"{trader}\nmedian={gaps.median():.0f}, max={gaps.max():.0f}", fontsize=8)
    ax.set_xlabel("Ticks between trades", fontsize=7)
    ax.set_ylabel("Count", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

for j in range(len(TRADERS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Waiting-Time Distribution Between Consecutive Trades", fontsize=13)
plt.tight_layout()
plt.show()

## 9 — Day-by-Day Stationarity Check

Do transition probabilities stay stable across the 3 days, or do traders change behaviour?

In [ ]:
fig, axes = plt.subplots(len(TRADERS), 3, figsize=(15, len(TRADERS) * 2.2))

for row, trader in enumerate(TRADERS):
    for col, day in enumerate([1, 2, 3]):
        ax = axes[row, col]
        tdf = activity[(activity["trader"] == trader) & (activity["day"] == day)]
        seq = tdf["side"].tolist()
        if len(seq) < 2:
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
            ax.set_visible(True)
            continue
        P, _ = build_transition_matrix(seq, SIDE_STATES)
        im = ax.imshow(P.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
        ax.set_xticks([0,1]); ax.set_xticklabels(SIDE_STATES, fontsize=7)
        ax.set_yticks([0,1]); ax.set_yticklabels(SIDE_STATES, fontsize=7)
        for i in range(2):
            for j in range(2):
                v = P.values[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                        color="white" if v > 0.6 else "black", fontweight="bold")
        if col == 0:
            ax.set_ylabel(trader, fontsize=8, rotation=0, labelpad=55, va="center")
        ax.set_title(f"Day {day}  (n={len(seq)})", fontsize=8)

fig.suptitle("Direction Markov per Day — Stationarity Check", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 10 — Pattern Summary & Trading Implications

In [ ]:
print("=" * 70)
print("PATTERN SUMMARY")
print("=" * 70)

for trader in TRADERS:
    P_dir = dir_results[trader]["P"]
    seq = activity[activity["trader"] == trader]
    n = len(seq)
    pct_buy  = 100 * (seq["side"] == "buy").mean()
    pct_sell = 100 - pct_buy

    pb_b = P_dir.loc["buy",  "buy"]
    ps_s = P_dir.loc["sell", "sell"]

    p_chi = dir_results[trader]["p"]
    sig = "*** SIGNIFICANT" if (p_chi and p_chi < 0.01) else ("* marginal" if (p_chi and p_chi < 0.1) else "")

    top_products = seq["product"].value_counts().head(3).index.tolist()

    print(f"\n{trader}  (n={n})")
    print(f"  Buy/Sell split : {pct_buy:.1f}% buy, {pct_sell:.1f}% sell")
    print(f"  P(buy|buy)     : {pb_b:.3f}   P(sell|sell): {ps_s:.3f}   {sig}")
    print(f"  Top products   : {top_products}")

    # Direction bias
    if pct_buy > 90:
        print("  >>> PURE BUYER — almost never sells")
    elif pct_sell > 90:
        print("  >>> PURE SELLER — almost never buys")
    elif pct_buy > 60:
        print("  >>> BUY-BIASED")
    elif pct_sell > 60:
        print("  >>> SELL-BIASED")
    else:
        print("  >>> BALANCED market-maker")

    # Persistence
    if pb_b > 0.9 or ps_s > 0.9:
        print("  >>> HIGH PERSISTENCE — tends to repeat the same side")
    elif pb_b < 0.3 or ps_s < 0.3:
        print("  >>> MEAN-REVERSION — frequently alternates sides")

print("\n" + "=" * 70)
print("KEY INSIGHT FOR TRADING")
print("=" * 70)
print("""
If you observe a Mark in your market_trades:

  Mark 01  — almost exclusively BUYS vouchers (VEV_5300-6500 range).
             Seeing Mark 01 buy → very likely to buy again next tick.
             Consider selling into him at a slight premium.

  Mark 22  — almost exclusively SELLS (primary liquidity provider / MM).
             Very persistent seller; likely posting offers across products.
             Can use as reference for fair ask prices.

  Mark 67  — EXCLUSIVELY BUYS. Never seen on the sell side across all 3 days.
             Highly predictable. When active, price likely to be pushed up.

  Mark 14  — Balanced. Likely a market-maker posting both sides.
             HYDROGEL_PACK and VELVETFRUIT_EXTRACT focused.

  Mark 38  — Balanced. Mix of HYDROGEL_PACK and VELVETFRUIT_EXTRACT.

  Mark 49  — Lightly active; more sell-side. Focused on VEV vouchers.

  Mark 55  — Balanced across delta-1 products.
""")